# Exercitiul 1 - Clasificare Binara (CU fisier adnotari + bbox)

**Varianta:** Imaginile contin personaje in context (nu doar fetele). Fisierul `adnotari.txt` contine:
```
nume_imagine xmin ymin xmax ymax nume_personaj
```
Trebuie sa **decupezi** fata conform bbox, apoi sa clasifici.

**Clase tipice:**
- Clasa 1 (pozitiv=1): ex. Tommy + Ora
- Clasa 0 (negativ=0): ex. Louie + Andy

**Scor:** acuratete × nr_puncte_maxim

In [ ]:
import os
import cv2
import numpy as np
from skimage.feature import hog
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
from copy import deepcopy
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# CONFIGURARE - MODIFICA DOAR ACEASTA SECTIUNE
# ============================================================

EX1_DIR      = "data/ex1"            # directorul cu imagini + adnotari.txt
ADNOTARI_TXT = "data/ex1/adnotari.txt"

# Numele personajelor din clasa pozitiva (1) si negativa (0)
CLASA_POZITIVA = ["Tommy", "Ora"]    # <-- modifica dupa enunt
CLASA_NEGATIVA = ["Louie", "Andy"]   # <-- modifica dupa enunt

# Cate imagini per personaj sunt de antrenare (restul -> testare)
# Enuntul specifica explicit: ex. primele 45 train, urm. 10 test
NR_TRAIN_PER_PERSONAJ = 45           # <-- modifica dupa enunt
# NR_TEST_PER_PERSONAJ se deduce automat

IMAGE_SIZE          = (128, 128)
HOG_ORIENTATIONS    = 9
HOG_PIX_PER_CELL    = (8, 8)
HOG_CELLS_PER_BLOCK = (3, 3)

In [ ]:
# ============================================================
# CITIRE ADNOTARI
# ============================================================
# Format linie: nume_img xmin ymin xmax ymax personaj

annotations = []  # lista de dict-uri

with open(ADNOTARI_TXT, 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) < 6:
            continue
        img_name, xmin, ymin, xmax, ymax, personaj = parts[0], int(parts[1]), int(parts[2]), int(parts[3]), int(parts[4]), parts[5]
        annotations.append({
            'img': img_name,
            'bbox': (xmin, ymin, xmax, ymax),
            'personaj': personaj
        })

print(f"Total adnotari: {len(annotations)}")
print("Primele 3:", annotations[:3])

In [ ]:
# ============================================================
# IMPARTIRE TRAIN / TEST
# Primele NR_TRAIN_PER_PERSONAJ imagini cu fiecare personaj -> train
# Restul -> test
# ============================================================

from collections import defaultdict

per_personaj = defaultdict(list)
for ann in annotations:
    per_personaj[ann['personaj']].append(ann)

train_anns = []
test_anns  = []

for personaj, anns in per_personaj.items():
    train_anns.extend(anns[:NR_TRAIN_PER_PERSONAJ])
    test_anns.extend(anns[NR_TRAIN_PER_PERSONAJ:])

print(f"Train: {len(train_anns)} | Test: {len(test_anns)}")

In [ ]:
# ============================================================
# INCARCARE IMAGINI + DECUPARE FATA + ETICHETE
# ============================================================

def get_label(personaj):
    if personaj in CLASA_POZITIVA:
        return 1
    elif personaj in CLASA_NEGATIVA:
        return 0
    else:
        return -1  # necunoscut - nu ar trebui sa apara

def load_face(ann, img_dir, size):
    """Incarca imaginea, decupeaza fata conform bbox, redimensioneaza."""
    img_path = os.path.join(img_dir, ann['img'])
    img = cv2.imread(img_path)
    if img is None:
        return None
    xmin, ymin, xmax, ymax = ann['bbox']
    # Clamp la dimensiunile imaginii
    h, w = img.shape[:2]
    xmin, ymin = max(0, xmin), max(0, ymin)
    xmax, ymax = min(w, xmax), min(h, ymax)
    face = img[ymin:ymax, xmin:xmax]
    if face.size == 0:
        return None
    face = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
    face = cv2.resize(face, size)
    return face

def load_dataset(anns, img_dir, size, augment=False):
    images, labels = [], []
    for ann in anns:
        face = load_face(ann, img_dir, size)
        if face is None:
            continue
        label = get_label(ann['personaj'])
        if label == -1:
            continue
        images.append(face)
        labels.append(label)
        if augment:  # flip orizontal ca augmentare
            images.append(cv2.flip(face, 1))
            labels.append(label)
    return images, labels

train_images, train_labels = load_dataset(train_anns, EX1_DIR, IMAGE_SIZE, augment=True)
test_images,  test_labels  = load_dataset(test_anns,  EX1_DIR, IMAGE_SIZE, augment=False)

print(f"Train: {len(train_images)} imagini | Test: {len(test_images)} imagini")
print(f"Train pozitive: {sum(train_labels)} | Train negative: {train_labels.count(0) if isinstance(train_labels, list) else (np.array(train_labels)==0).sum()}")

In [ ]:
# ============================================================
# EXTRAGERE CARACTERISTICI HOG
# ============================================================

def extract_hog(images):
    feats = []
    for img in images:
        fd = hog(
            img,
            orientations=HOG_ORIENTATIONS,
            pixels_per_cell=HOG_PIX_PER_CELL,
            cells_per_block=HOG_CELLS_PER_BLOCK,
            block_norm='L2-Hys',
            transform_sqrt=True,
            feature_vector=True
        )
        feats.append(fd)
    return np.array(feats, dtype=np.float32)

X_train = extract_hog(train_images)
X_test  = extract_hog(test_images)
y_train = np.array(train_labels)
y_test  = np.array(test_labels)

print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")

In [ ]:
# ============================================================
# ANTRENARE CLASIFICATOR (LinearSVC cu cautare C)
# ============================================================

best_model    = None
best_accuracy = 0

for C in [0.01, 0.1, 1.0, 10.0]:
    model = LinearSVC(C=C, max_iter=5000)
    model.fit(X_train, y_train)
    acc = model.score(X_train, y_train)
    print(f"C={C:.2f} -> train accuracy: {acc:.4f}")
    if acc > best_accuracy:
        best_accuracy = acc
        best_model = deepcopy(model)

print(f"\nBest train accuracy: {best_accuracy:.4f}")

In [ ]:
# ============================================================
# EVALUARE PE TEST
# ============================================================

y_pred = best_model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

# Vizualizare scoruri
scores = best_model.decision_function(X_train)
plt.figure(figsize=(10, 4))
plt.plot(np.sort(scores[y_train == 1]), label='Pozitive')
plt.plot(np.sort(scores[y_train == 0]), label='Negative')
plt.axhline(0, color='k', linestyle='--')
plt.legend()
plt.title('Distributia scorurilor pe train')
plt.show()